In [1]:
#Helper functions

def get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]): # iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        # if not at the very last position AND the pair matches, replace it
        if ids[i] == pair[0] and i < len(ids) - 1 and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [2]:
#byte-level BPE tokenizer

class BasicTokenizer:
    def train(self, text, vocab_size, verbose=False):
        text_bytes = text.encode('utf-8')
        ids = list(text_bytes) 
        num_merges = vocab_size - 256
        merges = {}
        vocab = {idx: bytes([idx]) for idx in range(256)}

        for i in range(num_merges):
            stats = get_stats(ids)
            pair = max(stats, key=stats.get)

            idx = 256+i
            ids = merge(ids, pair, idx)

            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")

        self.merges = merges
        self.vocab = vocab
                
    def decode(self, ids):
        text_bytes = b"".join(self.vocab[idx] for idx in ids)
        text = text_bytes.decode("utf-8", errors="replace")
        return text

    def encode(self, text):
        text_bytes = text.encode("utf-8") 
        ids = list(text_bytes) 
        while len(ids) >= 2:
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            idx = self.merges[pair]
            ids = merge(ids, pair, idx)
        return ids


In [3]:
import regex as re

GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

In [4]:
class RegexTokenizer:

    def __init__(self):
        self.pattern = GPT4_SPLIT_PATTERN

    def train(self, text, vocab_size, verbose=None):

        text_chunks = re.findall(self.pattern, text)
        
        text_bytes = [chunk.encode('utf-8') for chunk in text_chunks]
        ids = [list(b) for b in text_bytes]

        num_merges = vocab_size - 256
        merges = {} 
        vocab = {idx: bytes([idx]) for idx in range(256)}

        for i in range(num_merges):
            stats = {}
            for chunk_ids in ids:
                get_stats(chunk_ids, stats)

            if len(stats) == 0:
                break
            
            pair = max(stats, key = stats.get)
            idx = 256+i
            
            ids = [merge(chunk_ids, pair, idx) for chunk_ids in ids]
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")

        self.merges = merges
        self.vocab = vocab

    def decode(self, ids):
        #Given a list of integers we convert it to text
        tokens = b"".join(self.vocab[idx] for idx in ids)
        text = tokens.decode('utf-8', errors='replace')
        return text

    def encode_chunk(self, chunk_bytes):

        ids = list(chunk_bytes)
        
        while len(ids) >= 2:
            pairs = get_stats(ids)
            pair = min(pairs, key=lambda p: self.merges.get(p, float("inf")))

            if pair not in self.merges:
                break

            idx = self.merges[pair]
            ids = merge(ids, pair, idx)

        return ids
        
    def encode(self, text):
        #Given a string we convert it to list of integers based on what the tokenizer has learned
        text_chunks = re.findall(GPT4_SPLIT_PATTERN, text)
        text_bytes = [chunk.encode('utf-8') for chunk in text_chunks]
        ids = []
        
        for chunk_bytes in text_bytes:
            chunk_ids = self.encode_chunk(chunk_bytes)
            ids.extend(chunk_ids)

        return ids

In [19]:
tokenizer = RegexTokenizer()
with open('taylorswift.txt', 'r', encoding='utf-8') as f:
    text = f.read()
vocab_size = 500
tokenizer.train(text, vocab_size)

In [20]:
#Comparison with tiktoken
import tiktoken
text = 'hello world!!!? (안녕하세요!) lol123 😉'
enc = tiktoken.get_encoding('cl100k_base')
tokenizer.decode(tokenizer.encode(text)) == text, enc.decode(enc.encode(text)) == text

(True, True)

In [24]:
#Compression ratio
text2 = "Hello world, random text here"
ids = tokenizer.encode(text2)
chars_per_token = len(text2) / len(ids)
chars_per_token 

1.6111111111111112